# Phase offset demo
This demo shows how to extract phase offset measures between limb movements using data stored on huggingface hub. Phase offsets are computed by normalising movements within consencutive stance intervals.


In [ ]:
from pathlib import Path

from neurokinematics.pose.coordination import compute_phase_offset_pairs
from neurokinematics.pose.plotting import plot_phase_offset_pairs

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from huggingface_hub import hf_hub_download

## Download pose and stance data from huggingface hub


In [ ]:
data_dir = Path('example_pose_data')
data_dir.mkdir(exist_ok=True)

hugging_face_repo = 'cjblack1111/example_pose_data'
repo_type = 'dataset'
stance_fname = 'sko109120_baseline_stances.pkl'
pose_fname = 'sko109120_baseline_pose.csv'

# download data
stances_path = hf_hub_download(
    repo_id = hugging_face_repo,
    filename = stance_fname,
    repo_type = repo_type,
    local_dir = data_dir
)


pose_path = hf_hub_download(
    repo_id = hugging_face_repo,
    filename = pose_fname,
    repo_type = repo_type,
    local_dir = data_dir
)

Create a list of tuples containing the node pairs that will be used for calculating the phase offset, and load the pose and stance data into dataframes.

In [ ]:
node_pairs = [('r_forepaw', 'l_forepaw'), ('r_hindpaw', 'l_hindpaw')]

pose_df = pd.read_csv(pose_path)
stance_df = pd.read_pickle(stances_path)

## Compute phase offset
To compute the phase offset, pass the `pose_df` dataframe, `stance_df` dataframe, and `node_pairs` list to the `compute_phase_offset_pairs` function. This will return a dictionary with high-level keys for each tuple of node pairs, that contains `id`, `date`, `trial`, `values`, `locs`, `movement_ratio`, `max_speed`. The `values` key corresponds to the phase offset, which is given as a value between 0-1. 

In [ ]:
poff = compute_phase_offset_pairs(pose_df, stance_df, node_pairs)
print(f'poff dict keys: {poff.keys()}')
print(f'data for each pair: {poff[node_pairs[0]].keys()}')

To make plotting simple, use the helper function `plot_phase_offset_pairs` to plot the phase offset values for each node pair.

In [ ]:
plot_phase_offset_pairs(poff)

These plots can also be converted to plot as either radians or degrees with the `phase_mode` argument.

In [ ]:
plot_phase_offset_pairs(poff, phase_mode='degrees')
plot_phase_offset_pairs(poff, phase_mode='radians')

# Optional
These plots can be reproduced manually by accessing the phase offset values themselves.

In [ ]:
fig, ax  = plt.subplots(ncols=2)
bins = np.linspace(0,1,25)

for i, npair in enumerate(node_pairs):
    phase_ = np.concat(poff[npair]['values']) # concatenate phase offset values across trials/sessions
    ax[i].hist(phase_, bins, density=True)
    ax[i].set_xlabel('Phase offset')
    ax[i].set_ylabel('Density')
    ax[i].set_title(f'{npair[0]} - {npair[1]}')
    ax[i].set_ylim([0,3]) # setting limit for sample data

plt.tight_layout()
plt.show()